## `递归对象`
- 对象可以以其他的对象作为自己的属性值。当这个类下的对象实例有一个属性的值还属于这个类时，这个对象就是一个递归对象。 

### `2.类：链表`

In [1]:
class Link:
    """
    A linked list.
    """
    empty = ()
    def __init__(self, first, rest=()):
        assert rest == Link.empty or isinstance(rest, Link)
        self.first = first
        self.rest = rest
    
    def __getitem__(self, i):
        if i == 0:
            return self.first
        else:
            return self.rest[i - 1]
    
    def __len__(self):
        return 1 + len(self.rest)

In [9]:
s = Link(1, Link(2, Link(3)))
s[1]
len(s)

3

In [14]:
def link_expression(s):
    # 返回一个链表字符串
    if s.rest is Link.empty:
        rest = ''
    else:
        rest =  ', ' + link_expression(s.rest)
    return 'Link({0}{1})'.format(s.first, rest)

In [15]:
link_expression(s)

'Link(1, Link(2, Link(3)))'

In [16]:
Link.__repr__ = link_expression
s

Link(1, Link(2, Link(3)))

In [17]:
s_first = Link(s, Link(4))
s_first

Link(Link(1, Link(2, Link(3))), Link(4))

In [20]:
len(s_first)
len(s_first[0])
s_first[0][1]

2

In [21]:
def extend_link(s, t):
    if s is Link.empty:
        return t
    else:
        return Link(s.first, extend_link(s.rest, t))

In [23]:
extend_link(s, s)
Link.__add__ = extend_link
s + s

Link(1, Link(2, Link(3, Link(1, Link(2, Link(3))))))

In [24]:
def map_link(fn, s):
    if s is Link.empty:
        return s
    else:
        return Link(fn(s.first), map_link(fn, s.rest))

In [25]:
map_link(lambda x: x * x, s)

Link(1, Link(4, Link(9)))

In [26]:
def filter_link(fn, s):
    if s is Link.empty:
        return s
    else:
        rest = filter_link(fn, s.rest)
        if fn(s.first):
            return Link(s.first, rest)
        else:
            return rest
            

In [27]:
odd = lambda x: x % 2 == 1
filter_link(odd, s)

Link(1, Link(3))

In [28]:
def join_link(s, separator):
    if s is Link.empty:
        return ''
    elif s.rest is Link.empty:
        return str(s.first)
    else:
        return str(s.first) + separator + join_link(s.rest, separator)

join_link(s, ', ')

'1, 2, 3'

In [29]:
def partitions(n, m):
    """Return a linked list of partitions of n using parts of up to m.
    Each partition is represented as a linked list.
    """
    if n == 0:
        return Link(Link.empty)
    if n < 0 or m == 0:
        return Link.empty
    else:
        using_m = partitions(n - m, m)
        with_m = map_link(lambda s: Link(m, s), using_m)
        without_m = partitions(n, m - 1)
        return extend_link(with_m, without_m)

In [30]:
def print_partitions(n, m):
    lists = partitions(n, m)
    strings = map_link(lambda s: join_link(s, " + "), lists)
    print(join_link(strings, "\n"))

In [31]:
print_partitions(6, 4)

4 + 2
4 + 1 + 1
3 + 3
3 + 2 + 1
3 + 1 + 1 + 1
2 + 2 + 2
2 + 2 + 1 + 1
2 + 1 + 1 + 1 + 1
1 + 1 + 1 + 1 + 1 + 1


### `2.类：数`

In [32]:
class Tree:
    """
    A tree with label as its value and a list of branches.
    """
    def __init__(self, label, branches=()):
        self.label = label
        for branch in branches:
            assert isinstance(branch, Tree)
        self.branches = branches
    
    def __repr__(self):
        if self.branches:
            return "Tree({0}, {1})".format(self.label, repr(self.branches))
        else:
            return "Tree({0})".format(repr(self.label))
    
    def is_leaf(self):
        return not self.branches

In [38]:
def fib_tree(n):
    if n == 1:
        return Tree(0)
    elif n == 2:
        return Tree(1)
    else:
        left = fib_tree(n - 2)
        right = fib_tree(n - 1)
        return Tree(left.label + right.label, (left, right))

In [39]:
fib_tree(5)

Tree(3, (Tree(1, (Tree(0), Tree(1))), Tree(2, (Tree(1), Tree(1, (Tree(0), Tree(1)))))))

In [42]:
def sum_labels(t):
    return t.label + sum([sum_labels(b) for b in t.branches])

In [43]:
sum_labels(fib_tree(5))

10

In [47]:
from functools import cache 

fib_tree = cache(fib_tree)
big_fib_tree = fib_tree(35)
big_fib_tree.label

5702887

### `3.集合` 

In [48]:
s = {3, 2, 1, 4, 4}
s

{1, 2, 3, 4}

In [52]:
3 in s
len(s)
s.union({1, 5})
s.intersection({6, 5, 4, 3})

{3, 4}

In [53]:
def is_empty(s):
    return s is Link.empty

def set_contains(s, v):
    # 仅且仅当s包含v时返回true
    if is_empty(s):
        return False
    if s.first == v:
        return True
    else:
        return set_contains(s.rest, v)

In [54]:
s = Link(4, Link(1, Link(5)))

In [57]:
set_contains(s, 5)

True

In [61]:
def adjoin_set(s, v):
    # 返回一个包含s所有元素和v所有元素的集合
    if set_contains(s, v):
        return s
    else:
        return Link(v, s)

In [63]:
t = adjoin_set(s, 2)
t

Link(2, Link(4, Link(1, Link(5))))

In [64]:
def intersec_set(set1, set2):
    # 返回set1和set2的交集
    return filter_link(lambda v: set_contains(set2, v), set1)

In [69]:
square = lambda x: x * x
intersec_set(t, map_link(square, s))

Link(1)

In [71]:
"""
>>> def union_set(set1, set2):
        # 返回一个集合，包含 set1 和 set2 中的所有元素。
        set1_not_set2 = keep_if_link(set1, lambda v: not set_contains(set2, v))
        return extend_link(set1_not_set2, set2)
>>> union_set(t, s)
Link(2, Link(4, Link(1, Link(5))))
"""
def union_set(set1, set2):
    """返回一个集合, 包含set1和set2中的所有元素"""
    set1_not_set2 = filter_link(lambda v: not set_contains(set2, v), set1)
    return set1_not_set2 + set2

In [72]:
union_set(t, s)

Link(2, Link(4, Link(1, Link(5))))

In [73]:
t, s

(Link(2, Link(4, Link(1, Link(5)))), Link(4, Link(1, Link(5))))